# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).

**Nota de elaboración:** Las celdas Markdown cuentan con apoyo de inteligencia artificial generativa. Los ejemplos tienen fines educativos y demostrativos. El autor es responsable de su revisión, adaptación y validación en Microsoft Fabric.

# NB_02 — Extracción de información con `ai.extract`

**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo de referencia:** `gpt-5-mini`  
**Entorno:** Microsoft Fabric Notebook — PySpark  
**Tipo de endpoint:** endpoint LLM integrado de Fabric

## Objetivo

Aprender a utilizar `ai.extract()` para identificar y extraer información desde textos no estructurados almacenados en un DataFrame de Spark.

Este laboratorio cubre dos modalidades:

1. **Extracción por etiquetas:** obtener entidades como nombres, organizaciones y ubicaciones.
2. **Extracción estructurada:** definir tipos de datos y restricciones mediante `aifunc.ExtractLabel` y un esquema compatible con JSON Schema.

También consultaremos `ai.stats`, revisaremos los tokens y estimaremos el consumo de capacidad.

> **Importante:** la extracción es una inferencia de IA. Los valores obtenidos deben validarse antes de incorporarlos a procesos de negocio.

## Documentación oficial

- [Usar `ai.extract` con PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/extract)
- [AI Functions — descripción general](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)
- [Personalizar AI Functions con PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación de AI Functions](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)


## 1. Requisitos y consideraciones

Antes de ejecutar este notebook, verifica que:

- El workspace esté asignado a una capacidad Fabric compatible.
- El administrador haya habilitado la configuración de tenant necesaria para AI Functions.
- El notebook utilice un Fabric Runtime compatible con AI Functions.
- La sesión sea de tipo PySpark.

`ai.extract()` recibe una columna de texto y una o más etiquetas que representan los campos o entidades que se desean identificar. La función devuelve un nuevo DataFrame con columnas adicionales.

En PySpark, la transformación se distribuye a través de las particiones de Spark.

## 2. Configurar el modelo y las librerías

La documentación de PySpark utiliza el módulo `aifunc` desde `synapse.ml.spark`.

En este notebook se mantiene `gpt-5-mini` como modelo de referencia para facilitar la comparación con los otros laboratorios de la serie.

In [1]:
from synapse.ml.spark import aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"

aifunc.default_conf.set_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 3, Finished, Available, Finished, False)

Modelo configurado: gpt-5-mini


## 3. Crear datos ficticios

Utilizaremos descripciones ficticias de incidentes de soporte. El texto contiene información que normalmente se encontraría en correos, formularios o registros de atención.

Los datos incluyen nombre de contacto, organización, ciudad, aplicación, prioridad y cantidad de usuarios afectados.

No se utilizan datos personales reales.

In [2]:
data = [
    (
        1,
        "La solicitud fue reportada por Camila Torres de Contoso Chile, "
        "en Santiago. La aplicación afectada es Portal de Ventas. "
        "La prioridad es alta y hay 18 usuarios afectados."
    ),
    (
        2,
        "Andrés Molina, de Fabrikam, reporta desde Valparaíso una falla "
        "en el sistema ERP. El incidente tiene prioridad crítica y afecta "
        "a 42 usuarios."
    ),
    (
        3,
        "El contacto es Daniela Rojas de Northwind, ubicada en Concepción. "
        "Solicita soporte para Power BI. La prioridad es media y afecta "
        "a 6 usuarios."
    ),
    (
        4,
        "Javier Soto de Adventure Works, en Antofagasta, informa problemas "
        "con el acceso a Microsoft Teams. La prioridad es baja y afecta "
        "a 3 usuarios."
    ),
    (
        5,
        "María Pérez de Contoso Chile, desde Santiago, reporta que el "
        "sistema de inventario no permite registrar movimientos. "
        "La prioridad es alta y afecta a 11 usuarios."
    )
]

df = spark.createDataFrame(
    data,
    ["id_incidente", "descripcion"]
)

display(df)

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b1b52c6f-30b0-4bac-b3bc-4d8bd15a1c44)

## 4. Extracción básica mediante etiquetas

La forma más sencilla de utilizar `ai.extract()` consiste en indicar una lista de etiquetas.

### Sintaxis documentada

```python
df.ai.extract(
    labels=["name", "profession", "city"],
    input_col="descriptions"
)
```

En este ejemplo solicitaremos:

- `contact_name`: nombre de la persona de contacto.
- `organization`: organización mencionada.
- `city`: ciudad mencionada.

La salida contiene una columna por cada etiqueta. Por defecto, los valores se devuelven como listas de cadenas, porque puede existir más de una coincidencia por registro.

In [3]:
run_id_basic = str(uuid.uuid4())

resultado_basico = df.ai.extract(
    labels=[
        "contact_name",
        "organization",
        "city"
    ],
    input_col="descripcion",
    error_col="error_extraccion"
)

display(resultado_basico)

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b41cce7-fc60-4c31-b173-19d29eab1932)

## 5. Revisar la extracción básica

Revisaremos las entidades obtenidas y las filas que presenten errores.

Recuerda que una entidad no mencionada en el texto puede devolver `null`. La ausencia de un valor no necesariamente representa un error de ejecución.

In [4]:
display(
    resultado_basico.select(
        "id_incidente",
        "descripcion",
        "contact_name",
        "organization",
        "city",
        "error_extraccion"
    )
)

display(
    resultado_basico.filter(
        F.col("error_extraccion").isNotNull()
    )
)

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7ef4b087-47ba-4288-9025-ac8940f4e187)

SynapseWidget(Synapse.DataFrame, 20d2a054-650b-4bb7-8f7e-50afcedad265)

## 6. Extracción estructurada con `aifunc.ExtractLabel`

Además de extraer etiquetas simples, `ai.extract()` permite definir el tipo y las restricciones de los datos devueltos.

`aifunc.ExtractLabel` admite tipos de esquema como:

- `string`
- `number`
- `integer`
- `boolean`
- `object`
- `array`

También permite definir propiedades, enumeraciones (`enum`), campos obligatorios (`required`) y restringir propiedades adicionales (`additionalProperties=False`).

### Caso de uso

Extraeremos un objeto `incident_details` con estos campos:

| Campo | Tipo | Restricción |
|---|---|---|
| `application` | string | Nombre de la aplicación |
| `priority` | string | Enum: critical, high, medium, low |
| `affected_users` | integer | Cantidad de usuarios afectados |

Se definen los tres campos como obligatorios para mostrar cómo el esquema controla la estructura de la respuesta.

**Importante:** si el texto no contiene información suficiente para completar un campo obligatorio o el modelo no puede ajustarse al esquema, la fila puede generar una excepción. Esto se reflejará en `ai.stats`.


In [10]:
import inspect
from importlib.metadata import version

print("=" * 70)
print("DIAGNÓSTICO DE AI FUNCTIONS")
print("=" * 70)

print(f"Spark Runtime: {spark.version}")
print(f"SynapseML: {version('synapseml')}")

print("\nConstructor ExtractLabel:")
print(inspect.signature(aifunc.ExtractLabel))

print("\n" + "=" * 70)
print("MÉTODOS DISPONIBLES EN AI FUNCTIONS")
print("=" * 70)

for name in dir(df.ai):

    if name.startswith("_"):
        continue

    try:
        method = getattr(df.ai, name)

        if callable(method):
            print(f"\nMétodo: {name}")
            print(f"Firma: {inspect.signature(method)}")

    except Exception as e:
        print(f"No se pudo inspeccionar {name}: {e}")

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 12, Finished, Available, Finished, False)

DIAGNÓSTICO DE AI FUNCTIONS
Spark Runtime: 3.5.5.5.4.20260921.11
SynapseML: 1.1.3

Constructor ExtractLabel:
(label: str, *, description: Optional[str] = None, max_items: Optional[int] = None, type: str = 'string', properties: Optional[dict] = None)

MÉTODOS DISPONIBLES EN AI FUNCTIONS

Método: analyze_sentiment
Firma: (*, labels: Optional[list[str]] = None, input_col: str, output_col: str = None, error_col: str = None, input_col_type: str = 'text', **options) -> pyspark.sql.dataframe.DataFrame

Método: attach_stats
Firma: (results: pyspark.sql.dataframe.DataFrame) -> pyspark.sql.dataframe.DataFrame

Método: classify
Firma: (*, labels: list[str], input_col: str, output_col: str = None, error_col: str = None, input_col_type: str = 'text', **options) -> pyspark.sql.dataframe.DataFrame

Método: embed
Firma: (*, input_col: str, output_col: str = None, error_col: str = None, **options) -> pyspark.sql.dataframe.DataFrame

Método: extract
Firma: (*, labels: list[typing.Union[str, synapse.ml.u

In [8]:
# ============================================================
# EXTRACCIÓN ESTRUCTURADA CON AI FUNCTIONS
# ============================================================

incident_label = aifunc.ExtractLabel(
    label="incident_details",
    description=(
        "Extract the application name, incident priority and the number "
        "of affected users from the incident description. Do not infer "
        "values that are not present in the text."
    ),
    type="object",
    properties={
        "application": {
            "type": "string"
        },
        "priority": {
            "type": "string",
            "enum": [
                "critical",
                "high",
                "medium",
                "low"
            ]
        },
        "affected_users": {
            "type": "integer"
        }
    },
    max_items=1
)

# Identificador de ejecución para auditoría
run_id_structured = str(uuid.uuid4())

# Ejecutar extracción estructurada
resultado = df.ai.extract(
    labels=[incident_label],
    input_col="descripcion",
    error_col="error_extraccion"
)

display(resultado)

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea9f28d1-010e-4f3b-b235-3e4dcc633be7)

## 7. Inspeccionar el esquema de salida

La extracción estructurada permite que los campos tengan tipos definidos y que la respuesta cumpla las restricciones especificadas.

Inspeccionaremos el esquema real devuelto por Spark. La documentación indica que, al establecer `max_items=1`, el valor de la etiqueta se devuelve como una lista de un elemento; por eso es importante revisar el tipo de columna antes de transformar o aplanar la salida.

In [9]:
resultado.printSchema()

display(
    resultado.select(
        "id_incidente",
        "incident_details",
        "error_extraccion"
    )
)

# Conteo de filas con error
display(
    resultado
    .filter(F.col("error_extraccion").isNotNull())
    .select("id_incidente", "error_extraccion")
)

StatementMeta(, 2b956320-dfc2-4749-8912-55467667d642, 11, Finished, Available, Finished, False)

root
 |-- id_incidente: long (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- incident_details: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- error_extraccion: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = true)
 |    |    |    |-- minor: integer (nullable = true)
 |    |    |-- statusCode: integer (nullable = true)
 |    |    |-- reasonPhrase: string (nullable = true)



SynapseWidget(Synapse.DataFrame, f9c5d8d1-87bb-4786-9df1-94157f7b3f79)

SynapseWidget(Synapse.DataFrame, aeda6368-6b16-448c-a32a-553bda847565)

## 8. Consultar estadísticas con `ai.stats`

Las estadísticas se consultan desde el DataFrame resultante de la operación:

```python
stats_df = resultado.ai.stats
display(stats_df)
```

La documentación incluye métricas de ejecución y consumo de tokens, entre ellas `num_successful`, `num_exceptions`, `num_unevaluated`, `input_tokens`, `cached_tokens`, `output_tokens` y `reasoning_tokens`.

La disponibilidad de cada métrica depende de la respuesta del servicio. Por eso, el código consulta los campos de forma defensiva.

In [ ]:
stats_df = resultado.ai.stats

display(stats_df)

stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

stats = stats_rows[0].asDict(recursive=True)

print("===== ESTADÍSTICAS AI EXTRACT =====")
for key, value in stats.items():
    print(f"{key}: {value}")

## 9. Estimar el consumo de capacidad

Para `gpt-5-mini`, utilizaremos las tarifas publicadas para el endpoint integrado de Fabric:

| Tipo de token | CU-segundos por 1.000 tokens |
|---|---:|
| Entrada | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

La fórmula es:

$$
CU_s = \\frac{(I-C)\\times 8.40 + C\\times 0.84 + O\\times 67.23}{1000}
$$

Donde:

- \(I\): tokens de entrada.
- \(C\): tokens de entrada en caché.
- \(O\): tokens de salida.

**Alcance:** el cálculo es una estimación del consumo de las llamadas al modelo integrado. No representa el consumo total de Spark ni el consumo completo de capacidad del workspace.

Los tokens de razonamiento se muestran como dato informativo y no se agregan por separado a la fórmula.

In [ ]:
input_tokens = int(stats.get("input_tokens") or 0)
cached_tokens = int(stats.get("cached_tokens") or 0)
output_tokens = int(stats.get("output_tokens") or 0)
reasoning_tokens = int(stats.get("reasoning_tokens") or 0)

if cached_tokens > input_tokens:
    raise ValueError(
        "cached_tokens no puede ser mayor que input_tokens."
    )

INPUT_RATE = 8.40
CACHED_RATE = 0.84
OUTPUT_RATE = 67.23

regular_input_tokens = input_tokens - cached_tokens

estimated_cu_seconds = (
    regular_input_tokens * INPUT_RATE
    + cached_tokens * CACHED_RATE
    + output_tokens * OUTPUT_RATE
) / 1000

print("===== ESTIMACIÓN DE CONSUMO =====")
print(f"Modelo reportado: {stats.get('model', MODEL_NAME)}")
print(f"Tokens de entrada: {input_tokens:,}")
print(f"Tokens en caché: {cached_tokens:,}")
print(f"Tokens de entrada normales: {regular_input_tokens:,}")
print(f"Tokens de salida: {output_tokens:,}")
print(f"Tokens de razonamiento (informativo): {reasoning_tokens:,}")
print(f"CU-segundos estimados: {estimated_cu_seconds:.6f}")

## 10. Construir un registro de auditoría

Consolidaremos la ejecución de extracción estructurada en un registro de auditoría.

El registro contiene el identificador de ejecución, el modelo, el número de filas, las métricas de ejecución, los tokens y la estimación de CU-segundos.

La auditoría se genera en memoria. La persistencia en Delta queda como paso opcional.

In [ ]:
def metric_int(name):
    value = stats.get(name)
    return int(value) if value is not None else 0

num_input_rows = df.count()
num_successful = metric_int("num_successful")
num_exceptions = metric_int("num_exceptions")
num_unevaluated = metric_int("num_unevaluated")
num_harmful = metric_int("num_harmful")

if num_exceptions > 0 or num_harmful > 0:
    execution_status = "PARTIAL"
elif num_successful == num_input_rows:
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id_structured,
    datetime.now(timezone.utc).isoformat(),
    "NB_02_AI_EXTRACT",
    "ai.extract",
    MODEL_NAME,
    str(stats.get("model") or MODEL_NAME),
    num_input_rows,
    num_successful,
    num_exceptions,
    num_unevaluated,
    num_harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    float(estimated_cu_seconds),
    str(stats.get("client_type") or "unknown"),
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

## 11. Persistencia opcional en Lakehouse

Si deseas almacenar la auditoría en una tabla Delta del Lakehouse adjunto al notebook, puedes habilitar esta celda.

El modo `append` agrega un registro por ejecución. Verifica previamente el Lakehouse y el nombre de tabla.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 12. Extensión: extracción multimodal desde archivos

`ai.extract()` también puede trabajar con archivos, como PDF, imágenes y documentos de texto, utilizando una columna con rutas de archivos y el parámetro `input_col_type="path"`.

Esta funcionalidad se encuentra documentada en la experiencia de entrada multimodal, que actualmente se identifica como **Preview**. Requiere preparar correctamente las rutas y permisos de acceso a los archivos.

No ejecutamos esta modalidad en el laboratorio principal porque el objetivo aquí es aislar la extracción desde texto. Se desarrollará un notebook específico para procesamiento documental.

Documentación: [Use multimodal input with AI Functions](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview).

## 13. Conclusiones

En este laboratorio:

1. Configuramos AI Functions para PySpark.
2. Creamos un conjunto de incidentes ficticios.
3. Ejecutamos extracción básica mediante etiquetas.
4. Definimos una extracción estructurada con `aifunc.ExtractLabel`.
5. Revisamos el esquema de salida y errores.
6. Consultamos las estadísticas mediante `ai.stats`.
7. Estimamos CU-segundos a partir de los tokens reportados.
8. Generamos un registro de auditoría con persistencia opcional.

### Consideraciones para producción

- Definir cuidadosamente las etiquetas y descripciones de extracción.
- Utilizar esquemas tipados cuando los datos deban integrarse con sistemas posteriores.
- No marcar como obligatorios campos que legítimamente pueden faltar en el texto.
- Validar valores extraídos contra reglas de negocio y catálogos maestros.
- Revisar errores de esquema y excepciones en `ai.stats`.
- Medir el consumo real en Fabric Capacity Metrics y no confundirlo con la estimación basada en tokens.

## Referencias

- [Use ai.extract with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/extract)
- [AI Functions overview](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)
- [Customize AI Functions with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [AI Functions billing](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)
- [Use multimodal input with AI Functions (Preview)](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview)
